In [1]:
import pandas as pd
from src.utils.config import UHGG_EGGNOG_FOLDER, FINAL_LABELS_FILTERED, ALL_RM_LABELS, LABELS_RM_METABOLITES
from pathlib import Path

In [2]:
RM_SYSTEMS = {
    'type1': {
        'description': 'Type I R-M system (requires all three subunits)',
        'restriction': ['K01153'],  # hsdR; type I restriction enzyme, R subunit
        'specificity': ['K01154'],  # hsdS; type I restriction enzyme, S subunit
        'modification': ['K03427'],  # hsdM; type I restriction enzyme M protein
        'required_logic': 'all_three'
    },

    'type2': {
        'description': 'Type II R-M system (requires restriction enzyme + any methylase)',
        'restriction': [
            'K01155'  # type II restriction enzyme
        ],
        'modification': [
            'K00558',  # DNMT1, dcm; DNA (cytosine-5)-methyltransferase 1
            'K00571',  # site-specific DNA-methyltransferase (adenine-specific)
            'K00590',  # site-specific DNA-methyltransferase (cytosine-N4-specific)
            'K06223',  # dam; DNA adenine methylase
            'K07317',  # adenine-specific DNA-methyltransferase
            'K07318',  # adenine-specific DNA-methyltransferase
            'K07319',  # yhdJ; adenine-specific DNA-methyltransferase
            'K13581',  # ccrM; modification methylase
            'K24451'   # zim; modification methylase
        ],
        'required_logic': 'restriction_plus_any_modification'
    },

    'type3': {
        'description': 'Type III R-M system (requires both Res and Mod subunits)',
        'restriction': ['K01156'],  # res; type III restriction enzyme
        'modification': ['K07316'],  # mod; adenine-specific DNA-methyltransferase
        'required_logic': 'both_required'
    },
}

In [3]:
def kos_to_labels(species_ko_set):
    # Type I: Need all three subunits
    has_type1 = (
        'K01153' in species_ko_set and  # R subunit
        'K01154' in species_ko_set and  # S subunit
        'K03427' in species_ko_set      # M subunit
    )

    # Type II: Need restriction enzyme + at least one methylase
    has_type2_restriction = 'K01155' in species_ko_set

    type2_methylases = {
        'K00558', 'K00571', 'K00590', 'K06223',
        'K07317', 'K07318', 'K07319', 'K13581', 'K24451'
    }
    has_type2_modification = bool(species_ko_set & type2_methylases)

    has_type2 = has_type2_restriction and has_type2_modification

    # Type III: Need both Res and Mod subunits
    has_type3 = (
        'K01156' in species_ko_set and  # Res subunit
        'K07316' in species_ko_set      # Mod subunit
    )

    # Overall R-M capability (Types I-III only, unless Type IV included)
    has_any_rm = has_type1 or has_type2 or has_type3

    result = {
        'type1': int(has_type1),
        'type2': int(has_type2),
        'type3': int(has_type3),
        'any_rm': int(has_any_rm),
    }


    return result

In [4]:
def get_species_rm_labels(genome_id: str):
    filename = f"{genome_id}_eggNOG.tsv"
    filepath = Path(UHGG_EGGNOG_FOLDER / genome_id / filename)
    if not filepath.exists():
        return None

    annotations_tsv = pd.read_csv(filepath, sep="\t")
    clean_tsv = annotations_tsv[annotations_tsv['KEGG_ko']!='-'].copy()
    all_kos = set()
    try:
        for index, row in clean_tsv.iterrows():
            kegg_ko = row['KEGG_ko']
            for ko in kegg_ko.strip().split(','):
                if ':' in ko:
                    _, K0 = ko.split(":")
                    all_kos.add(K0)
        labels = {
            "genome_id": genome_id
        }
        labels.update(kos_to_labels(all_kos))
    except Exception as e:
        print(f"Error processing {genome_id}: {e}")
        return None
    return labels

In [5]:
final_labels_csv = pd.read_csv(FINAL_LABELS_FILTERED)
species_ids = final_labels_csv['genome_id'].tolist()

all_rm_labels = []
for species in species_ids:
    result = get_species_rm_labels(species)
    if result is not None:
        all_rm_labels.append(result)

rm_labels_df = pd.DataFrame(all_rm_labels)
rm_labels_df.rename(columns={"any_rm": "rm"}, inplace=True)
rm_labels_df.to_csv(ALL_RM_LABELS, index=False)
rm_labels_df

,genome_id,type1,type2,type3,rm
0,MGYG000000001,1,0,0,1
1,MGYG000000002,1,0,0,1
2,MGYG000000003,1,0,0,1
3,MGYG000000004,1,1,1,1
4,MGYG000000005,1,0,0,1
...,...,...,...,...,...
4695,MGYG000004901,0,0,0,0
4696,MGYG000004902,1,0,0,1
4697,MGYG000004903,1,0,0,1
4698,MGYG000004904,0,0,0,0


In [6]:
labels = final_labels_csv.merge(rm_labels_df, on="genome_id")
labels = labels.drop('type2', axis=1)
if 'Unnamed: 0' in labels.columns:
    labels = labels.drop('Unnamed: 0', axis=1)

labels.to_csv(LABELS_RM_METABOLITES, index=False)
labels

,genome_id,b12,heme,folate,biotin,type1,type3,rm
0,MGYG000000001,1,0,1,0,1,0,1
1,MGYG000000002,1,0,1,0,1,0,1
2,MGYG000000003,0,0,1,0,1,0,1
3,MGYG000000004,1,0,0,0,1,1,1
4,MGYG000000005,1,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...
4695,MGYG000004901,1,0,0,1,0,0,0
4696,MGYG000004902,1,0,0,0,1,0,1
4697,MGYG000004903,1,0,0,0,1,0,1
4698,MGYG000004904,0,0,0,0,0,0,0


In [7]:
print(f"Total species: {len(labels)}")
print(f"\nR-M System Prevalence:")
print(f"  Any R-M: {labels['rm'].sum()} ({labels['rm'].mean() * 100:.1f}%)")
print(f"  Type I:  {labels['type1'].sum()} ({labels['type1'].mean() * 100:.1f}%)")
print(f"  Type III: {labels['type3'].sum()} ({labels['type3'].mean() * 100:.1f}%)")

# Cross-tabulation
print(f"\nSpecies with multiple R-M types:")
labels['n_rm_types'] = labels['type1']  + labels['type3']
print(labels['n_rm_types'].value_counts().sort_index())

Total species: 4700

R-M System Prevalence:
  Any R-M: 2340 (49.8%)
  Type I:  1609 (34.2%)
  Type III: 838 (17.8%)

Species with multiple R-M types:
n_rm_types
0    2632
1    1689
2     379
Name: count, dtype: int64
